# 04_02: Visual Question Answering (VQA)

**Objective:** Learn to build systems that answer natural language questions about images by combining vision and language understanding.

**Prerequisites:** Familiarity with HuggingFace pipelines (Notebook 00_03), image classification concepts (Notebook 02_01).

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|-------------|------------|------|---------|-------------------|-------|
| **Small (CPU)** | dandelin/vilt-b32-finetuned-vqa | ~440MB | 4GB RAM | Any CPU | ViLT model, fast |
| **Large (GPU)** | Salesforce/blip-vqa-base | ~990MB | 8GB RAM | RTX 4080+ | BLIP, higher accuracy |

## Expected Behaviors

### First Time Running
- **Model download**: ~440MB for ViLT (2-3 minutes)
- Image downloads from URLs are small (~100KB each)

### What You'll See
- Models answer open-ended questions about image content (objects, colors, counts, actions)
- ViLT processes image patches and text tokens together in a single Transformer
- Answers are typically short (1-3 words) for classification-based VQA models

### Common Observations
- Factual questions ("What color is the car?") get higher confidence than subjective ones
- Counting questions are challenging for most VQA models
- Questions about small or occluded objects may yield incorrect answers

## Overview

**Visual Question Answering (VQA)** combines computer vision and natural language processing: given an image and a question about that image, the model produces an answer.

### VQA Approaches

| Approach | Description | Example Models |
|----------|-------------|---------------|
| **Classification** | Select from ~3,000 pre-defined answers | ViLT, LXMERT |
| **Generative** | Generate free-form text answers | BLIP, BLIP-2, GIT |

Classification-based models are faster but limited to known answer types. Generative models can produce any text but are larger and slower.

### How ViLT Works

```
Image → Split into patches → Linear projection → Patch embeddings
Text  → Tokenize          → Token embeddings
                                    ↓
         Concatenate [patch_embs; text_embs]
                                    ↓
              Single Transformer Encoder
                                    ↓
              Classification head → Answer
```

## Setup and Installation

In [ ]:
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import (
    pipeline,
    ViltProcessor,
    ViltForQuestionAnswering,
)
from PIL import Image
import matplotlib.pyplot as plt
import requests
from io import BytesIO

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Version metadata
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Model Selection

In [ ]:
# Option 1: Small Model (CPU-friendly, classification-based)
MODEL_NAME = 'dandelin/vilt-b32-finetuned-vqa'   # ~440MB, 3129 answer classes

# Option 2: Large Model (GPU-optimized, generative)
# MODEL_NAME = 'Salesforce/blip-vqa-base'         # ~990MB, free-form answers

## Helper Functions

In [ ]:
def load_image_from_url(url: str, max_size: int = 512) -> Image.Image:
    """Load and resize an image from a URL.

    Args:
        url: URL of the image.
        max_size: Maximum dimension.

    Returns:
        PIL Image object.
    """
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    image = Image.open(BytesIO(response.content)).convert('RGB')
    if max(image.size) > max_size:
        ratio = max_size / max(image.size)
        new_size = (int(image.size[0] * ratio), int(image.size[1] * ratio))
        image = image.resize(new_size, Image.LANCZOS)
    return image


# Load test images
IMAGE_URLS = {
    'cats': 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg',
    'football': 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/image_qa.jpg',
}

images: dict[str, Image.Image] = {}
for name, url in IMAGE_URLS.items():
    try:
        images[name] = load_image_from_url(url)
        print(f'Loaded "{name}": {images[name].size}')
    except Exception as error:
        print(f'Could not load "{name}": {error}')

if not images:
    print('Creating synthetic fallback image...')
    images['synthetic'] = Image.fromarray(
        np.random.randint(0, 255, (384, 512, 3), dtype=np.uint8)
    )

## Method 1: Pipeline API

The VQA pipeline takes an image and a question, then returns the top predicted answers with confidence scores.

In [ ]:
# Create VQA pipeline
vqa_pipeline = pipeline(
    'visual-question-answering',
    model=MODEL_NAME,
    device=device,
)

# Pick the first available image
test_image_name = list(images.keys())[0]
test_image = images[test_image_name]

# Show the image
plt.figure(figsize=(6, 5))
plt.imshow(test_image)
plt.title(f'Test Image: {test_image_name}', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Ask questions about the image
questions = [
    'What animal is in the image?',
    'What color is it?',
    'How many animals are there?',
    'Is it indoors or outdoors?',
    'What is the animal doing?',
]

results: list[dict[str, str]] = []
for question in questions:
    answers = vqa_pipeline(
        image=test_image,
        question=question,
        top_k=3,
    )
    top = answers[0]
    results.append({
        'Question': question,
        'Answer': top['answer'],
        'Score': f'{top["score"]:.4f}',
        'Runner-up': answers[1]['answer'] if len(answers) > 1 else 'N/A',
    })

pd.DataFrame(results)

The model returns short answers from its vocabulary of ~3,129 pre-defined classes. This classification approach is fast but cannot generate novel or complex answers. Notice how confidence varies — factual questions about visible attributes tend to get higher scores than interpretive questions.

## Method 2: Manual Model Loading

Loading the model manually lets us inspect the raw logits, see all candidate answers ranked by probability, and understand the model's internal representations.

In [ ]:
# Load ViLT model and processor
processor = ViltProcessor.from_pretrained(MODEL_NAME)
model = ViltForQuestionAnswering.from_pretrained(MODEL_NAME).to(device)
model.eval()

print(f'Model: {MODEL_NAME}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Number of answer classes: {model.config.num_labels}')

In [ ]:
def vqa_manual(
    image: Image.Image,
    question: str,
    processor: ViltProcessor,
    model: torch.nn.Module,
    top_k: int = 5,
) -> pd.DataFrame:
    """Answer a visual question using manual model inference.

    Args:
        image: Input PIL image.
        question: Question about the image.
        processor: ViLT processor.
        model: ViLT VQA model.
        top_k: Number of top answers to return.

    Returns:
        DataFrame with top-k answers and scores.
    """
    inputs = processor(image, question, return_tensors='pt').to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits[0]
    probs = torch.sigmoid(logits)  # VQA uses sigmoid, not softmax
    top_probs, top_indices = torch.topk(probs, top_k)
    
    rows: list[dict[str, str]] = []
    for prob, idx in zip(top_probs, top_indices):
        rows.append({
            'Rank': str(len(rows) + 1),
            'Answer': model.config.id2label[idx.item()],
            'Score': f'{prob.item():.4f}',
        })
    return pd.DataFrame(rows)


# Get detailed top-5 answers
question = 'What animal is in the image?'
print(f'Question: "{question}"\n')
vqa_manual(test_image, question, processor, model, top_k=5)

VQA models use **sigmoid** instead of softmax because multiple answers can be correct simultaneously (e.g., an animal could be both "cat" and "kitten"). The scores represent independent probabilities rather than a distribution that sums to 1.

## Multi-Image Comparison

Let's ask the same set of questions about different images to see how the model adapts its answers to visual content.

In [ ]:
def compare_images_qa(
    images_dict: dict[str, Image.Image],
    questions: list[str],
    vqa_pipe: transformers.Pipeline,
) -> pd.DataFrame:
    """Ask the same questions about multiple images and compare answers.

    Args:
        images_dict: Dict mapping image names to PIL images.
        questions: List of questions to ask.
        vqa_pipe: VQA pipeline.

    Returns:
        DataFrame with answers per image for each question.
    """
    rows: list[dict[str, str]] = []
    for question in questions:
        row: dict[str, str] = {'Question': question}
        for img_name, img in images_dict.items():
            answer = vqa_pipe(image=img, question=question, top_k=1)
            row[img_name] = f'{answer[0]["answer"]} ({answer[0]["score"]:.2f})'
        rows.append(row)
    return pd.DataFrame(rows)


if len(images) >= 2:
    comparison_questions = [
        'What is in the image?',
        'What color is most prominent?',
        'Is it indoors or outdoors?',
        'How many objects are there?',
    ]
    print('=== Cross-Image QA Comparison ===')
    compare_images_qa(images, comparison_questions, vqa_pipeline)
else:
    print('Only one image available. Load more images for comparison.')

## Practical Applications

VQA has applications in accessibility (describing images for visually impaired users), content moderation (checking image content against policies), e-commerce (answering product questions from images), and medical imaging (preliminary diagnosis support).

In [ ]:
# Application: Image content audit
def audit_image(
    image: Image.Image,
    audit_questions: list[str],
    vqa_pipe: transformers.Pipeline,
) -> pd.DataFrame:
    """Run a content audit on an image using predefined questions.

    Args:
        image: PIL image to audit.
        audit_questions: List of audit questions.
        vqa_pipe: VQA pipeline.

    Returns:
        DataFrame with audit results.
    """
    results: list[dict[str, str]] = []
    for question in audit_questions:
        answers = vqa_pipe(image=image, question=question, top_k=2)
        top = answers[0]
        confidence = 'High' if top['score'] > 0.7 else 'Medium' if top['score'] > 0.3 else 'Low'
        results.append({
            'Audit Question': question,
            'Answer': top['answer'],
            'Confidence': confidence,
            'Score': f'{top["score"]:.4f}',
        })
    return pd.DataFrame(results)


audit_questions = [
    'Are there any people in the image?',
    'Is there any text in the image?',
    'What is the main subject?',
    'Is the image appropriate for children?',
    'What time of day is it?',
]

print('=== Image Content Audit ===')
audit_image(test_image, audit_questions, vqa_pipeline)

## Performance Benchmarking

In [ ]:
def benchmark_vqa(
    image: Image.Image,
    questions: list[str],
    vqa_pipe: transformers.Pipeline,
    num_runs: int = 5,
) -> pd.DataFrame:
    """Benchmark VQA inference speed.

    Args:
        image: Test image.
        questions: Questions to benchmark with.
        vqa_pipe: VQA pipeline.
        num_runs: Number of runs for averaging.

    Returns:
        DataFrame with timing results.
    """
    # Warmup
    vqa_pipe(image=image, question=questions[0], top_k=1)
    
    times: list[float] = []
    for _ in range(num_runs):
        start = time.perf_counter()
        for q in questions:
            vqa_pipe(image=image, question=q, top_k=1)
        times.append(time.perf_counter() - start)
    
    return pd.DataFrame([{
        'Model': MODEL_NAME.split('/')[-1],
        'Questions': len(questions),
        'Total Time (ms)': f'{np.mean(times) * 1000:.1f}',
        'Per Question (ms)': f'{np.mean(times) * 1000 / len(questions):.1f}',
        'Device': str(device),
    }])


bench_questions = ['What is this?', 'What color?', 'How many?', 'Where is it?', 'Is it large?']
benchmark_vqa(test_image, bench_questions, vqa_pipeline)

## Exercises

1. **BLIP comparison**: Load `Salesforce/blip-vqa-base` (if you have GPU/RAM) and compare its answers with ViLT on the same images. BLIP generates free-form text — how do the answer styles differ?

2. **Adversarial questions**: Ask questions about things that aren't in the image. Does the model confidently give wrong answers, or does it indicate uncertainty?

3. **Custom images**: Use your own photos and ask 10+ questions. Track which categories of questions (color, count, object, action, spatial) the model handles best.

4. **VQA dataset**: Load the `Graphcore/vqa` dataset from HuggingFace and evaluate the model on a sample of 50 questions. Compute accuracy.

## Key Takeaways

- **VQA** combines vision and language — the model takes an image + question and returns an answer
- **Classification-based VQA** (ViLT) selects from ~3,129 pre-defined answers using sigmoid activations
- **Generative VQA** (BLIP) can produce free-form text answers but requires more compute
- VQA models use **sigmoid** (not softmax) because multiple answers can be simultaneously valid
- Factual, visible-attribute questions work best; counting and spatial reasoning remain challenging

## Next Steps & Resources

**Next notebook**: [04_03 — Text-to-Image Generation](04_03_multimodal_text_to_image.ipynb) — generate images from text prompts.

**Resources:**
- [ViLT Paper](https://arxiv.org/abs/2102.03334) — Vision-and-Language Transformer
- [BLIP Paper](https://arxiv.org/abs/2201.12086) — Bootstrapping Language-Image Pre-training
- [VQA Dataset](https://visualqa.org/) — Standard VQA benchmark
- [VQA Models on Hub](https://huggingface.co/models?pipeline_tag=visual-question-answering)